### Fig 2B : Time Traj , GMM Hist for selected networks

In [ ]:
ENV["OMP_NUM_THREADS"] = "1"

using HDF5
using Plots
using Measures
using PyCall
using Statistics
using LinearAlgebra
using StatsBase
using LaTeXStrings

gr()

# =======================================================
# File Paths
# =======================================================
traj_file = raw"F:\HR_Project\3nets_fullres\multimodal_search_3nets_fullres_V3.h5"
hist_file = raw"F:\HR_Project\3nets_fullres\multimodal_search_3nets_fullres_V3.h5"

# =======================================================
# Configs
# =======================================================
configs = [
    (net="net_1762", param="param_72", ic="ic_1"), 
    (net="net_31",   param="param_96", ic="ic_1"), 
    (net="net_1982", param="param_400", ic="ic_1"),
]

target_trials = ["trial_1", "trial_2"]
node_idx      = 1

# =======================================================
# Plot style
# =======================================================
const TITLE_SIZE  = 30
const LABEL_SIZE  = 30
const TICK_SIZE   = 25
const LEGEND_SIZE = 25
const LINE_W_TRAJ = 3.5
const LINE_W_HIST = 0.0
const ALPHA_UNI   = 0.85
const ALPHA_BI    = 0.75
const COLOR_GREY  = :gray45
const COLOR_BI_1  = "#2166AC"
const COLOR_BI_2  = "#D6604D"
const TRAJ_XLIMS  = (1000.0, 1800.0)
const TRAJ_COLORS = [:hotpink, :forestgreen]
const BOX_THICKNESS = 2.0

function panel_kw(;
        title  = "",
        xlabel = "",
        ylabel = "",
        xlims  = nothing,
        ylims  = nothing,
)
    kw = (
        title                   = title,
        xlabel                  = xlabel,
        ylabel                  = ylabel,
        framestyle              = :box,
        grid                    = false,
        minorticks              = true,
        legend                  = :best,
        titlefont               = font(TITLE_SIZE, "Helvetica Bold"),
        guidefont               = font(LABEL_SIZE, "Helvetica Bold"),
        tickfont                = font(TICK_SIZE,  "Helvetica Bold"),
        legendfont              = font(LEGEND_SIZE,"Helvetica Bold"),
        tick_direction = :out,
        #tick_direction = :none,
        tick_length    = 10,
        #thickness_scaling       = 1/0.5,
        left_margin             = 8mm,
        right_margin            = 4mm,
        top_margin              = 5mm,
        bottom_margin           = 8mm,
        foreground_color_border = :black,
        foreground_color_axis   = :black,
        background_color        = :white,
        xguidefontcolor         = :black,
        yguidefontcolor         = :black,
        xtickfontcolor          = :black,
        ytickfontcolor          = :black,
    )
    xlims !== nothing && (kw = merge(kw, (xlims=xlims,)))
    ylims !== nothing && (kw = merge(kw, (ylims=ylims,)))
    return kw
end

function ticks_length!(p, idx; tl=0.03, mtl=0.015)
    # Get Major Ticks from the plot object
    xt_major = Plots.xticks(p)[idx][1]
    yt_major = Plots.yticks(p)[idx][1]
    
    xl = Plots.xlims(p[idx])
    yl = Plots.ylims(p[idx])
    sz = p.attr[:size]
    aspect = sz[1] / sz[2]

    # --- YOUR MINOR TICK LOGIC ---
    xt_minor = Float64[]; yt_minor = Float64[]
    if length(xt_major) ≥ 2
        for i in 1:length(xt_major)-1
            append!(xt_minor, range(xt_major[i], xt_major[i+1], length=6)[2:5])
        end
    end
    if length(yt_major) ≥ 2
        for i in 1:length(yt_major)-1
            append!(yt_minor, range(yt_major[i], yt_major[i+1], length=6)[2:5])
        end
    end

    # Scale lengths
    dx_maj = tl * (xl[2] - xl[1]);     dy_maj = tl * aspect * (yl[2] - yl[1])
    dx_min = mtl * (xl[2] - xl[1]);    dy_min = mtl * aspect * (yl[2] - yl[1])

    # Plot Major Ticks
    plot!(p[idx], [xt_major xt_major]', [fill(yl[1], length(xt_major)) fill(yl[1]+dy_maj, length(xt_major))]';
          c=:black, lw=2.5, label=false)
    plot!(p[idx], [fill(xl[1], length(yt_major)) fill(xl[1]+dx_maj, length(yt_major))]', [yt_major yt_major]';
          c=:black, lw=2.5, label=false)

    # Plot Minor Ticks
    if !isempty(xt_minor)
        plot!(p[idx], [xt_minor xt_minor]', [fill(yl[1], length(xt_minor)) fill(yl[1]+dy_min, length(xt_minor))]';
              c=:black, lw=2.0, label=false)
    end
    if !isempty(yt_minor)
        plot!(p[idx], [fill(xl[1], length(yt_minor)) fill(xl[1]+dx_min, length(yt_minor))]', [yt_minor yt_minor]';
              c=:black, lw=2.0, label=false)
    end

    plot!(p[idx], xlims=xl, ylims=yl) # Lock limits
end
# =======================================================
# 1. TRAJECTORY 
# =======================================================
function get_global_traj_ylims(traj_file, configs, trials, node)
    ymin, ymax = Inf, -Inf
    h5open(traj_file, "r") do file
        for c in configs, t_key in trials
            path = "$(c.net)/$(c.param)/$(c.ic)/$t_key"
            haskey(file, path) || continue
            d    = read(file[path])
            traj = size(d,1) < size(d,2) ? d[node,:] : d[:,node]
            ymin = min(ymin, minimum(traj))
            ymax = max(ymax, maximum(traj))
        end
    end
    pad = (ymax - ymin) * 0.12
    return (ymin - pad, ymax + pad)
end

function fill_traj_panel!(p, idx, traj_file, net, param, ic,
                            trials, node, ylims_traj)
    first_series = true
    h5open(traj_file, "r") do file
        base = "$net/$param/$ic"
        haskey(file, base) || return
        for (i, tk) in enumerate(trials)
            full = "$base/$tk"
            haskey(file, full) || continue
            d    = read(file[full])
            traj = size(d,1) < size(d,2) ? d[node,:] : d[:,node]
            t    = collect(range(0.0, 2000.0, length=length(traj)))
            lbl  = i == 1 ? "Cell A" : "Cell B"
            if first_series
                plot!(p[idx], t, traj;
                    label  = lbl,
                    color  = TRAJ_COLORS[i],
                    lw     = LINE_W_TRAJ,
                    panel_kw(
                        xlabel = "Time (A.U.)",
                        ylabel = "X$(node) Level",
                        xlims  = TRAJ_XLIMS,
                        ylims  = ylims_traj,
                    )...
                )
                first_series = false
            else
                plot!(p[idx], t, traj;
                    label = lbl,
                    color = TRAJ_COLORS[i],
                    lw    = LINE_W_TRAJ,
                )
            end
        end
    end
end

# =======================================================
# 2. HISTOGRAM 
# =======================================================
function species_bimodal_flags(gmm, data_final; gap_threshold=0.20)
    means = gmm.means_
    ns    = size(data_final, 2)
    flags = fill(false, ns)
    for d in 1:ns
        col = data_final[:, d]
        rng = maximum(col) - minimum(col)
        rng < 1e-12 && continue
        gap      = abs(means[1,d] - means[2,d]) / rng
        flags[d] = gap >= gap_threshold
    end
    return flags
end

function load_and_fit(hist_file, net, param; gap_threshold=0.20)
    sklearn = pyimport("sklearn.mixture")
    init_pts, final_pts = Vector{Vector{Float64}}(), Vector{Vector{Float64}}()
    h5open(hist_file, "r") do file
        key = "$net/$param"
        haskey(file, key) || error("$key not found")
        g = file[key]
        for ik in keys(g)
            startswith(ik,"ic_") || continue
            for tk in keys(g[ik])
                startswith(tk,"trial_") || continue
                d = read(g[ik][tk])
                size(d,2) < 2 && continue
                push!(init_pts,  vec(d[:,2]))
                push!(final_pts, vec(d[:,end]))
            end
        end
    end
    data_initial = reduce(hcat, init_pts)'
    data_final   = reduce(hcat, final_pts)'
    
    gmm1 = sklearn.GaussianMixture(n_components=1, covariance_type="full", n_init=5, random_state=42).fit(data_final)
    gmm2 = sklearn.GaussianMixture(n_components=2, covariance_type="full", n_init=5, random_state=42).fit(data_final)
    bic1, bic2 = gmm1.bic(data_final), gmm2.bic(data_final)

    if bic2 < bic1
        sflags = species_bimodal_flags(gmm2, data_final; gap_threshold)
        if any(sflags)
            best_k, best_gmm = 2, gmm2
        else
            best_k, best_gmm = 1, gmm1
            sflags = fill(false, size(data_final,2))
        end
    else
        best_k, best_gmm = 1, gmm1
        sflags = fill(false, size(data_final,2))
    end

    labels = best_gmm.predict(data_final)
    return data_initial, data_final, labels, best_k, sflags
end

function get_global_hist_xlims(hist_results, node)
    all_vals = Float64[]
    for r in hist_results
        append!(all_vals, r.data_initial[:, node])
        append!(all_vals, r.data_final[:,  node])
    end
    xpad = 0.05*(maximum(all_vals)-minimum(all_vals))
    return (minimum(all_vals)-xpad, maximum(all_vals)+xpad)
end

function fill_hist_panel!(p, idx, data_vec, cluster_labels, best_k,
                            is_bimodal, title_str, xlims)
    common = (
        bins      = 50,
        normalize = :pdf,
        linewidth = 0,
        linecolor = :match,
        xlims     = xlims,
        panel_kw(
            title  = title_str,
            xlabel = "X1 Level",
            ylabel = "Density", 
            xlims  = xlims,
        )...
    )

    if best_k == 1 || !is_bimodal
        histogram!(p[idx], collect(data_vec);
            color = COLOR_GREY, alpha = ALPHA_UNI,
            label = "Unimodal", common...)
    else
        histogram!(p[idx], collect(data_vec[cluster_labels .== 0]);
            color = COLOR_BI_1, alpha = ALPHA_BI,
            label = "State 1", common...)
            
        histogram!(p[idx], collect(data_vec[cluster_labels .== 1]);
            color     = COLOR_BI_2, 
            alpha     = ALPHA_BI,
            label     = "State 2",
            bins      = 50, 
            normalize = :pdf, 
            linewidth = 0,
            linecolor = :match)
    end
end

# =======================================================
# 3. COMBINED 3×3 FIGURE
# =======================================================
function build_combined_3x3(configs, traj_file, hist_file,
                              node; savepath="combined_3x3_high_visibility.svg")

    n_rows = length(configs)
    n_cols = 3

    ylims_traj = get_global_traj_ylims(traj_file, configs, target_trials, node)

    hist_results = []
    for c in configs
        d_i, d_f, clabels, bk, sf = load_and_fit(hist_file, c.net, c.param)
        push!(hist_results, (
            net=c.net, param=c.param,
            data_initial=d_i, data_final=d_f,
            cluster_labels=clabels, best_k=bk, sflags=sf,
        ))
    end

    xlims_hist = get_global_hist_xlims(hist_results, node)

    p = plot(
        layout           = (n_rows, n_cols),
        size             = (650*n_cols, 550*n_rows),
        dpi              = 600,
        background_color = :white,
    )

    for (row, (c, hr)) in enumerate(zip(configs, hist_results))
        # Define indices correctly
        idx_traj  = (row-1)*n_cols + 1
        idx_early = (row-1)*n_cols + 2
        idx_final = (row-1)*n_cols + 3

        is_bimodal = (hr.best_k == 2) && hr.sflags[node]

        t_early = (row == 1) ? "Time = 50" : ""
        t_final = (row == 1) ? "Time = 2000" : ""

        # Column 1: Trajectories
        fill_traj_panel!(p, idx_traj, traj_file, c.net, c.param, c.ic,
                         target_trials, node, ylims_traj)

        # Column 2: Initial Distribution
        fill_hist_panel!(p, idx_early, collect(hr.data_initial[:, node]),
                         hr.cluster_labels, hr.best_k, false,
                         t_early, xlims_hist)

        # Column 3: Final Distribution
        fill_hist_panel!(p, idx_final, collect(hr.data_final[:, node]),
                         hr.cluster_labels, hr.best_k, is_bimodal,
                         t_final, xlims_hist)

        # Draw custom tick marks on all three panels in this row
        ticks_length!(p, idx_traj;  tl=0.035, mtl=0.015)
        ticks_length!(p, idx_early; tl=0.035, mtl=0.015)
        ticks_length!(p, idx_final; tl=0.035, mtl=0.015)
    end

    display(p)
    savefig(p, savepath)
    return p
end

# =======================================================
# EXECUTION
# =======================================================
p_out = build_combined_3x3(configs, traj_file, hist_file, node_idx)
println("\n=========== ANALYSIS COMPLETE ===========")

### Fig 2C : Contigency table & hist for FB, w/o FB

In [ ]:
###############################################################
# Full Analysis: Multimodality vs Feedback Fisher's Test , OR
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using Plots
using HypothesisTests

println("Starting analysis...")

# ------------------------------------------------------------
# 1. Load multimodal CSV
# ------------------------------------------------------------
csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1.csv"
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

println("Loaded multimodal data.")

# ------------------------------------------------------------
# 2. Load all networks from JLD2
# ------------------------------------------------------------
dir = raw"F:\HR_Project\noise1"
jld_path = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]
all_net_ids = collect(keys(all_matrices))

println("Loaded network matrices.")

# ------------------------------------------------------------
# 3. Function to detect feedback
# ------------------------------------------------------------
function has_feedback(mat)
    # 1. Self-loops
    for i in 1:3
        if mat[i,i] != 0
            return true
        end
    end
    # 2. Pairwise loops
    pairs = [(1,2),(1,3),(2,3)]
    for (i,j) in pairs
        if mat[i,j] != 0 && mat[j,i] != 0
            return true
        end
    end
    # 3. 3-node cycles
    clockwise = mat[1,2]!=0 && mat[2,3]!=0 && mat[3,1]!=0
    counter = mat[1,3]!=0 && mat[3,2]!=0 && mat[2,1]!=0
    if clockwise || counter
        return true
    end
    return false
end

# ------------------------------------------------------------
# 4. Filter out networks that are all-zero
# ------------------------------------------------------------
filtered_ids = [id for id in all_net_ids if any(all_matrices[id].adj_matrix .!= 0)]
all_nets_df = DataFrame(net_id = filtered_ids)

# Merge with CSV counts
net_counts_full = leftjoin(all_nets_df, net_counts, on=:net_id)

# Replace missing counts and percentages with 0
net_counts_full.count = coalesce.(net_counts_full.count, 0)
net_counts_full.percentage_multimodal = coalesce.(net_counts_full.percentage_multimodal, 0.0)

# Add feedback info
net_counts_full.feedback = [has_feedback(all_matrices[id].adj_matrix) for id in net_counts_full.net_id]

# ------------------------------------------------------------
# 5. Compute summary counts
# ------------------------------------------------------------
n_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== true))
n_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== false))
n_non_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== true))
n_non_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== false))

println("\nContingency Table Counts:")
println("Multimodal + FB      = ", n_multimodal_with_fb)
println("Multimodal + No FB   = ", n_multimodal_no_fb)
println("Non-Multimodal + FB  = ", n_non_multimodal_with_fb)
println("Non-Multimodal + NoFB= ", n_non_multimodal_no_fb)

# ------------------------------------------------------------
# 6. Build 2x2 Table
# ------------------------------------------------------------
a = n_multimodal_with_fb
b = n_multimodal_no_fb
c = n_non_multimodal_with_fb
d = n_non_multimodal_no_fb

table = [a b; c d]
println("\n2x2 Table:")
println(table)

# ------------------------------------------------------------
# 7. Odds Ratio (Haldane-Anscombe correction if zero)
# ------------------------------------------------------------
# Original counts (integers)
a_int = n_multimodal_with_fb
b_int = n_multimodal_no_fb
c_int = n_non_multimodal_with_fb
d_int = n_non_multimodal_no_fb

# Odds ratio (apply 0.5 correction if zero) Haldane-Anscombe correction
a_or, b_or, c_or, d_or = a_int, b_int, c_int, d_int
if any([a_int,b_int,c_int,d_int] .== 0)
    a_or += 0.5
    b_or += 0.5
    c_or += 0.5
    d_or += 0.5
end

OR = (a_or*d_or)/(b_or*c_or)
logOR = log(OR)

println("\nOdds Ratio = ", OR)
println("Log Odds Ratio = ", logOR)

# Fisher test (use integer counts!)
fet = FisherExactTest(a_int, b_int, c_int, d_int)
p_two = pvalue(fet)               # two-sided
p_one = pvalue(fet, tail=:right)  # one-sided

println("\nFisher Test Results:")
println("One-sided p-value  = ", p_one)
println("Two-sided p-value  = ", p_two)

# ------------------------------------------------------------
# 9. Separate percentages for plotting
# ------------------------------------------------------------
with_fb = net_counts_full.percentage_multimodal[net_counts_full.feedback .== true]
without_fb = net_counts_full.percentage_multimodal[net_counts_full.feedback .== false]

# ------------------------------------------------------------
# 10. Plot stacked histogram
# ------------------------------------------------------------
plot = histogram(
    [with_fb, without_fb],
    label = ["With Feedback" "Without Feedback"],
    color = [:teal :orange],
    bins = 30,
    title = "Overall Network Multimodality",
    xlabel = "Percentage Multimodal",
    ylabel = "Number of Networks",
    legend = :topright,
    alpha = 0.7,
    lw = 0.5,
    xlims = (0, 100),
    ylims = (0, 3000),
    normalize = false
)

println("\nAnalysis complete.")
display(plot)

In [ ]:
using Plots
using Colors

Plots.default(dpi=600)

# =========================================================
# DATA
# =========================================================
counts = [2976   0;
           414  20]

p_val = "< 0.0001"
odds_ratio = 294.418

# =========================================================
# LOG TRANSFORM 
# =========================================================
log_counts = log10.(counts .+ 1)

# =========================================================
# AXES
# =========================================================
x = [1, 2]
y = [1, 2]

# =========================================================
# HEATMAP
# =========================================================
h = heatmap(
    x,
    y,
    log_counts,

    c = :YlGnBu, 
    colorbar = true,
    colorbar_pad = 15,  

    framestyle = :box,
    grid = false,
    linewidth = 2.0,
    aspect_ratio = 1,
    size = (750, 600),

    xlims = (0.5, 2.5),
    ylims = (0.5, 2.5),

    xticks = (x, ["With FB", "Without FB"]),
    yticks = (y, ["HR-Enabling \nNetworks", "HR-Disabling \nNetworks"]),
    yflip = true,

    xtickfont  = font(14, "Helvetica Bold"),
    ytickfont  = font(14, "Helvetica Bold"),
    xguidefont = font(18, "Helvetica Bold"),
    yguidefont = font(18, "Helvetica Bold"),
    legendfont = font(12, "Helvetica Bold"),

    left_margin   = 25Plots.mm,
    right_margin  = 20Plots.mm,   
    top_margin    = 15Plots.mm,
    bottom_margin = 15Plots.mm,

    colorbar_title = "\nlog10(count + 1)"
)

# =========================================================
# CELL ANNOTATIONS 
# =========================================================
# Text colors configured to dynamically contrast against YlGnBu tiles
annotate!(1, 1, text("n = 2976", 12, :white, :bold))
annotate!(2, 1, text("n = 0", 12, :black, :bold))

annotate!(1, 2, text("n = 414", 12, :white, :bold))
annotate!(2, 2, text("n = 20", 12, :black, :bold))

# =========================================================
# DISPLAY & SAVE
# =========================================================
display(h)
savefig(h, "contingency_heatmap_FB.png")
savefig(h, "contingency_heatmap_FB.svg")

In [ ]:
using Plots
using Colors
gr()
Plots.default(dpi=600)

# ------------------------------------------------------------
# 1. Data
# ------------------------------------------------------------
hr_with_fbl    = 2976
hr_wo_fbl      = 0
hr_total       = hr_with_fbl + hr_wo_fbl

nonhr_with_fbl = 414
nonhr_wo_fbl   = 20
nonhr_total    = nonhr_with_fbl + nonhr_wo_fbl

hr_pct_fbl     = hr_with_fbl    / hr_total    * 100   
nonhr_pct_fbl  = nonhr_with_fbl / nonhr_total * 100   

# ------------------------------------------------------------
# 2. Manual horizontal stacked bars (draw order = paint order)
# ------------------------------------------------------------
col_fbl    = colorant"#F08080"  # w/ FBL  (red)
col_no_fbl = colorant"#808080"  # w/o FBL (gray)

bar_h   = 0.6   # bar thickness
y_hr    = 2.0   # y-center for HR bar (top)
y_nonhr = 1.0   # y-center for Non-HR bar (bottom)

function hbar_segment(y, x0, x1, h, color)
    Shape([x0, x1, x1, x0], [y-h/2, y-h/2, y+h/2, y+h/2])
end

p = plot(
    legend = false,
    xlims  = (0, 100),
    ylims  = (0.3, 2.7),
    yticks = ([y_nonhr, y_hr], ["Non-HR", "HR"]),
    xticks = ([0, 100], ["0%", "100%"]),
    framestyle = :box,
    grid = false,
    size = (500, 400),
    xtickfont  = font(14, "Helvetica"),
    ytickfont  = font(14, "Helvetica"),
    bottom_margin = 5Plots.mm,
    left_margin   = 8Plots.mm,
    right_margin   = 8Plots.mm,
    xmirror = true   
)

# HR bar: red first , then gray 
plot!(p, hbar_segment(y_hr, 0, hr_pct_fbl, bar_h, col_fbl),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(p, hbar_segment(y_hr, hr_pct_fbl, 100, bar_h, col_no_fbl),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

# Non-HR bar: red first, then gray on top 
plot!(p, hbar_segment(y_nonhr, 0, nonhr_pct_fbl, bar_h, col_fbl),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(p, hbar_segment(y_nonhr, nonhr_pct_fbl, 100, bar_h, col_no_fbl),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

display(p)
savefig(p, "figure_2C_style.png")

### Fig 2D : Contigency table & hist for PFB, w/o PFB

In [ ]:
###############################################################
# Full Analysis: Multimodality vs Positive Feedback Fisher's Test , OR
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using Plots
using HypothesisTests
using Colors

gr()
Plots.default(dpi=600)

println("Starting analysis...")

# ------------------------------------------------------------
# 1. Load multimodal CSV
# ------------------------------------------------------------
csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1.csv"
df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

println("Loaded multimodal data.")

# ------------------------------------------------------------
# 2. Load all networks from JLD2
# ------------------------------------------------------------
dir = raw"F:\HR_Project\noise1"
jld_path = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]
all_net_ids = collect(keys(all_matrices))

println("Loaded network matrices.")

# ------------------------------------------------------------
# 3. Function to detect positive feedback loops specifically
# ------------------------------------------------------------
function has_positive_feedback(mat)
    # 1. Self-loops (must be auto-activation to be positive)
    for i in 1:3
        if mat[i,i] > 0
            return true
        end
    end
    # 2. Pairwise loops (product of signs must be positive: +/+ or -/-)
    pairs = [(1,2),(1,3),(2,3)]
    for (i,j) in pairs
        if mat[i,j] != 0 && mat[j,i] != 0
            if (mat[i,j] * mat[j,i]) > 0
                return true
            end
        end
    end
    # 3. 3-node cycles (product of all three signs must be positive)
    if mat[1,2] != 0 && mat[2,3] != 0 && mat[3,1] != 0
        if (mat[1,2] * mat[2,3] * mat[3,1]) > 0
            return true
        end
    end
    if mat[1,3] != 0 && mat[3,2] != 0 && mat[2,1] != 0
        if (mat[1,3] * mat[3,2] * mat[2,1]) > 0
            return true
        end
    end
    return false
end

# ------------------------------------------------------------
# 4. Filter out networks that are all-zero and merge
# ------------------------------------------------------------
filtered_ids = [id for id in all_net_ids if any(all_matrices[id].adj_matrix .!= 0)]
all_nets_df = DataFrame(net_id = filtered_ids)

# Merge with CSV counts
net_counts_full = leftjoin(all_nets_df, net_counts, on=:net_id)

# Replace missing counts and percentages with 0
net_counts_full.count = coalesce.(net_counts_full.count, 0)
net_counts_full.percentage_multimodal = coalesce.(net_counts_full.percentage_multimodal, 0.0)

# Add positive feedback info
net_counts_full.feedback = [has_positive_feedback(all_matrices[id].adj_matrix) for id in net_counts_full.net_id]

# ------------------------------------------------------------
# 5. Compute summary counts
# ------------------------------------------------------------
n_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== true))
n_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== false))
n_non_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== true))
n_non_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== false))

println("\nContingency Table Counts (Positive FB):")
println("Multimodal + Pos FB      = ", n_multimodal_with_fb)
println("Multimodal + No Pos FB   = ", n_multimodal_no_fb)
println("Non-Multimodal + Pos FB  = ", n_non_multimodal_with_fb)
println("Non-Multimodal + No PosFB= ", n_non_multimodal_no_fb)

# ------------------------------------------------------------
# 6. Build 2x2 Table
# ------------------------------------------------------------
a = n_multimodal_with_fb
b = n_multimodal_no_fb
c = n_non_multimodal_with_fb
d = n_non_multimodal_no_fb

table = [a b; c d]
println("\n2x2 Table:")
println(table)

# ------------------------------------------------------------
# 7. Odds Ratio (Haldane-Anscombe correction if zero)
# ------------------------------------------------------------
a_int = n_multimodal_with_fb
b_int = n_multimodal_no_fb
c_int = n_non_multimodal_with_fb
d_int = n_non_multimodal_no_fb

a_or, b_or, c_or, d_or = a_int, b_int, c_int, d_int
if any([a_int,b_int,c_int,d_int] .== 0)
    a_or += 0.5
    b_or += 0.5
    c_or += 0.5
    d_or += 0.5
end

OR = (a_or*d_or)/(b_or*c_or)
logOR = log(OR)

println("\nOdds Ratio = ", OR)
println("Log Odds Ratio = ", logOR)

# Fisher test (using integer counts)
fet = FisherExactTest(a_int, b_int, c_int, d_int)
p_two = pvalue(fet)               
p_one = pvalue(fet, tail=:right)  

println("\nFisher Test Results:")
println("One-sided p-value  = ", p_one)
println("Two-sided p-value  = ", p_two)

# ------------------------------------------------------------
# 8. Manual Horizontal Stacked Bar Plot 
#     HR = Multimodal, Non-HR = Non-Multimodal
#     Uses the Fisher's test contingency counts a, b, c, d
# ------------------------------------------------------------
hr_total    = a + b   # multimodal total
nonhr_total = c + d   # non-multimodal total

hr_pct_fbl    = hr_total    == 0 ? 0.0 : (a / hr_total)    * 100   # multimodal w/ pos FB
nonhr_pct_fbl = nonhr_total == 0 ? 0.0 : (c / nonhr_total) * 100   # non-multimodal w/ pos FB

col_fbl    = colorant"#F08080"  # w/ Pos FBL  (red)
col_no_fbl = colorant"#808080"  # w/o Pos FBL (gray)

bar_h   = 0.6   # bar thickness
y_hr    = 2.0   # y-center for HR (Multimodal) bar (top)
y_nonhr = 1.0   # y-center for Non-HR (Non-Multimodal) bar (bottom)

function hbar_segment(y, x0, x1, h)
    Shape([x0, x1, x1, x0], [y-h/2, y-h/2, y+h/2, y+h/2])
end

plot2 = plot(
    legend = false,
    xlims  = (0, 100),
    ylims  = (0.3, 2.7),
    yticks = ([y_nonhr, y_hr], ["Non-HR", "HR"]),
    xticks = ([0, 100], ["0%", "100%"]),
    framestyle = :box,
    grid = false,
    size = (500, 400),
    xtickfont  = font(14, "Helvetica"),
    ytickfont  = font(14, "Helvetica"),
    bottom_margin = 5Plots.mm,
    left_margin   = 8Plots.mm,
    right_margin   = 8Plots.mm,
    xmirror = true   
)

# HR (Multimodal) bar: red first ,then gray 
plot!(plot2, hbar_segment(y_hr, 0, hr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_hr, hr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

# Non-HR (Non-Multimodal) bar: red first, then gray on top
plot!(plot2, hbar_segment(y_nonhr, 0, nonhr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_nonhr, nonhr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

display(plot2)

# ------------------------------------------------------------
# 9. SAVE Horizontal Stacked Bar Plot
# ------------------------------------------------------------
savefig(plot2, "figure_2D_style_PosFB.svg")
savefig(plot2, "figure_2D_style_PosFB.png")

println("Analysis complete. Stacked horizontal bar plot saved.")

In [ ]:
using Plots
using Colors

Plots.default(dpi=600)

# =========================================================
# DATA
# =========================================================
counts = [2711   265;
           312  122]

p_val = "< 0.0001"
odds_ratio = 4.0

# =========================================================
# LOG TRANSFORM 
# =========================================================
log_counts = log10.(counts .+ 1)

# =========================================================
# AXES
# =========================================================
x = [1, 2]
y = [1, 2]

# =========================================================
# HEATMAP
# =========================================================
h = heatmap(
    x,
    y,
    log_counts,

    c = :YlGnBu, # Native sequential Yellow-Green-Blue color palette
    colorbar = true,
    colorbar_pad = 15,  

    framestyle = :box,
    grid = false,
    linewidth = 2.0,
    aspect_ratio = 1,
    size = (750, 600),

    xlims = (0.5, 2.5),
    ylims = (0.5, 2.5),

    xticks = (x, ["With PosFB", "Without PosFB"]),
    yticks = (y, ["HR-Enabling \nNetworks", "HR-Disabling \nNetworks"]),
    yflip = true,

    xtickfont  = font(14, "Helvetica Bold"),
    ytickfont  = font(14, "Helvetica Bold"),
    xguidefont = font(18, "Helvetica Bold"),
    yguidefont = font(18, "Helvetica Bold"),
    legendfont = font(12, "Helvetica Bold"),

    left_margin   = 25Plots.mm,
    right_margin  = 20Plots.mm,   
    top_margin    = 15Plots.mm,
    bottom_margin = 15Plots.mm,

    colorbar_title = "\nlog10(count + 1)"
)

# =========================================================
# CELL ANNOTATIONS 
# =========================================================
# Text colors configured to dynamically contrast against YlGnBu tiles
annotate!(1, 1, text("n = 2711", 12, :white, :bold))
annotate!(2, 1, text("n = 265", 12, :black, :bold))

annotate!(1, 2, text("n = 312", 12, :black, :bold))
annotate!(2, 2, text("n = 122", 12, :black, :bold))

# =========================================================
# DISPLAY & SAVE
# =========================================================
display(h)
savefig(h, "contingency_heatmap_PosFB.png")
savefig(h, "contingency_heatmap_PosFB.svg")

### Fig 2E : Contigency table & hist for NFB, w/o NFB

In [ ]:
###############################################################
# Full Analysis: Multimodality vs Negative Feedback Fisher's Test, OR
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using Plots
using HypothesisTests
using Colors

gr()
Plots.default(dpi=600)

println("Starting analysis...")

# ------------------------------------------------------------
# 1. Load multimodal CSV
# ------------------------------------------------------------
csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1.csv"
df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

println("Loaded multimodal data.")

# ------------------------------------------------------------
# 2. Load all networks from JLD2
# ------------------------------------------------------------
dir = raw"F:\HR_Project\noise1"
jld_path = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]
all_net_ids = collect(keys(all_matrices))

println("Loaded network matrices.")

# ------------------------------------------------------------
# 3. Function to detect negative feedback loops specifically
# ------------------------------------------------------------
function has_negative_feedback(mat)
    # 1. Self-loops (must be auto-inhibition to be negative)
    for i in 1:3
        if mat[i,i] < 0
            return true
        end
    end
    # 2. Pairwise loops (product of signs must be negative, e.g., +/- or -/+)
    pairs = [(1,2),(1,3),(2,3)]
    for (i,j) in pairs
        if mat[i,j] != 0 && mat[j,i] != 0
            if (mat[i,j] * mat[j,i]) < 0
                return true
            end
        end
    end
    # 3. 3-node cycles (product of all three signs must be negative)
    if mat[1,2] != 0 && mat[2,3] != 0 && mat[3,1] != 0
        if (mat[1,2] * mat[2,3] * mat[3,1]) < 0
            return true
        end
    end
    if mat[1,3] != 0 && mat[3,2] != 0 && mat[2,1] != 0
        if (mat[1,3] * mat[3,2] * mat[2,1]) < 0
            return true
        end
    end
    return false
end

# ------------------------------------------------------------
# 4. Filter out networks that are all-zero and merge
# ------------------------------------------------------------
filtered_ids = [id for id in all_net_ids if any(all_matrices[id].adj_matrix .!= 0)]
all_nets_df = DataFrame(net_id = filtered_ids)

# Merge with CSV counts
net_counts_full = leftjoin(all_nets_df, net_counts, on=:net_id)

# Replace missing counts and percentages with 0
net_counts_full.count = coalesce.(net_counts_full.count, 0)
net_counts_full.percentage_multimodal = coalesce.(net_counts_full.percentage_multimodal, 0.0)

# Add negative feedback info
net_counts_full.feedback = [has_negative_feedback(all_matrices[id].adj_matrix) for id in net_counts_full.net_id]

# ------------------------------------------------------------
# 5. Compute summary counts
# ------------------------------------------------------------
n_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== true))
n_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .> 0) .& (net_counts_full.feedback .== false))
n_non_multimodal_with_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== true))
n_non_multimodal_no_fb = sum((net_counts_full.percentage_multimodal .== 0) .& (net_counts_full.feedback .== false))

println("\nContingency Table Counts (Negative FB):")
println("Multimodal + Neg FB      = ", n_multimodal_with_fb)
println("Multimodal + No Neg FB   = ", n_multimodal_no_fb)
println("Non-Multimodal + Neg FB  = ", n_non_multimodal_with_fb)
println("Non-Multimodal + No NegFB= ", n_non_multimodal_no_fb)

# ------------------------------------------------------------
# 6. Build 2x2 Table
# ------------------------------------------------------------
a = n_multimodal_with_fb
b = n_multimodal_no_fb
c = n_non_multimodal_with_fb
d = n_non_multimodal_no_fb

table = [a b; c d]
println("\n2x2 Table:")
println(table)

# ------------------------------------------------------------
# 7. Odds Ratio (Haldane-Anscombe correction if zero)
# ------------------------------------------------------------
a_int = n_multimodal_with_fb
b_int = n_multimodal_no_fb
c_int = n_non_multimodal_with_fb
d_int = n_non_multimodal_no_fb

a_or, b_or, c_or, d_or = a_int, b_int, c_int, d_int
if any([a_int,b_int,c_int,d_int] .== 0)
    a_or += 0.5
    b_or += 0.5
    c_or += 0.5
    d_or += 0.5
end

OR = (a_or*d_or)/(b_or*c_or)
logOR = log(OR)

println("\nOdds Ratio = ", OR)
println("Log Odds Ratio = ", logOR)

# Fisher test (using integer counts)
fet = FisherExactTest(a_int, b_int, c_int, d_int)
p_two = pvalue(fet)               
p_one = pvalue(fet, tail=:right)  

println("\nFisher Test Results:")
println("One-sided p-value  = ", p_one)
println("Two-sided p-value  = ", p_two)

# ------------------------------------------------------------
# 8. Manual Horizontal Stacked Bar Plot 
#     HR = Multimodal, Non-HR = Non-Multimodal
#     Uses the Fisher's test contingency counts a, b, c, d
# ------------------------------------------------------------
hr_total    = a + b   # multimodal total
nonhr_total = c + d   # non-multimodal total

hr_pct_fbl    = hr_total    == 0 ? 0.0 : (a / hr_total)    * 100   # multimodal w/ neg FB
nonhr_pct_fbl = nonhr_total == 0 ? 0.0 : (c / nonhr_total) * 100   # non-multimodal w/ neg FB

col_fbl    = colorant"#F08080"  # w/ Neg FBL  (red)
col_no_fbl = colorant"#808080"  # w/o Neg FBL (gray)

bar_h   = 0.6   # bar thickness
y_hr    = 2.0   # y-center for HR (Multimodal) bar (top)
y_nonhr = 1.0   # y-center for Non-HR (Non-Multimodal) bar (bottom)

function hbar_segment(y, x0, x1, h)
    Shape([x0, x1, x1, x0], [y-h/2, y-h/2, y+h/2, y+h/2])
end

plot2 = plot(
    legend = false,
    xlims  = (0, 100),
    ylims  = (0.3, 2.7),
    yticks = ([y_nonhr, y_hr], ["Non-HR", "HR"]),
    xticks = ([0, 100], ["0%", "100%"]),
    framestyle = :box,
    grid = false,
    size = (500, 400),
    xtickfont  = font(14, "Helvetica"),
    ytickfont  = font(14, "Helvetica"),
    bottom_margin = 5Plots.mm,
    left_margin   = 8Plots.mm,
    right_margin   = 8Plots.mm,
    xmirror = true  
)

# HR (Multimodal) bar: red first, then gray 
plot!(plot2, hbar_segment(y_hr, 0, hr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_hr, hr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

# Non-HR (Non-Multimodal) bar: red first, then gray on top
plot!(plot2, hbar_segment(y_nonhr, 0, nonhr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_nonhr, nonhr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

display(plot2)

# ------------------------------------------------------------
# 9. SAVE Horizontal Stacked Bar Plot
# ------------------------------------------------------------
savefig(plot2, "figure_2E_style_NegFB.svg")
savefig(plot2, "figure_2E_style_NegFB.png")

println("Analysis complete. Stacked horizontal bar plot saved.")

In [ ]:
using Plots
using Colors

Plots.default(dpi=600)

# =========================================================
# DATA
# =========================================================
counts = [2693   283;
           300  134]

p_val = "< 0.0001"
odds_ratio = 4.25

# =========================================================
# LOG TRANSFORM 
# =========================================================
log_counts = log10.(counts .+ 1)

# =========================================================
# AXES
# =========================================================
x = [1, 2]
y = [1, 2]

# =========================================================
# HEATMAP
# =========================================================
h = heatmap(
    x,
    y,
    log_counts,

    c = :YlGnBu, # Native sequential Yellow-Green-Blue color palette
    colorbar = true,
    colorbar_pad = 15,  

    framestyle = :box,
    grid = false,
    linewidth = 2.0,
    aspect_ratio = 1,
    size = (750, 600),

    xlims = (0.5, 2.5),
    ylims = (0.5, 2.5),

    xticks = (x, ["With NegFB", "Without NegFB"]),
    yticks = (y, ["HR-Enabling \nNetworks", "HR-Disabling \nNetworks"]),
    yflip = true,

    xtickfont  = font(14, "Helvetica Bold"),
    ytickfont  = font(14, "Helvetica Bold"),
    xguidefont = font(18, "Helvetica Bold"),
    yguidefont = font(18, "Helvetica Bold"),
    legendfont = font(12, "Helvetica Bold"),

    left_margin   = 25Plots.mm,
    right_margin  = 20Plots.mm,   
    top_margin    = 15Plots.mm,
    bottom_margin = 15Plots.mm,

    colorbar_title = "\nlog10(count + 1)"
)

# =========================================================
# CELL ANNOTATIONS 
# =========================================================
# Text colors configured to dynamically contrast against YlGnBu tiles
annotate!(1, 1, text("n = 2693", 12, :white, :bold))
annotate!(2, 1, text("n = 283", 12, :black, :bold))

annotate!(1, 2, text("n = 300", 12, :black, :bold))
annotate!(2, 2, text("n = 134", 12, :black, :bold))

# =========================================================
# DISPLAY & SAVE
# =========================================================
display(h)
savefig(h, "contingency_heatmap_NegFB.png")
savefig(h, "contingency_heatmap_NegFB.svg")

### Fig 2F 2G : Edgewise analysis for PFB and NFB

In [ ]:
###############################################################
# Pos FB + Neg FB Combined
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using HypothesisTests
using Printf
using Plots
using Measures

gr()

Plots.default(
    xtickfont  = font(22, "Helvetica Bold"),
    ytickfont  = font(22, "Helvetica Bold"),
    xguidefont = font(24, "Helvetica Bold"),
    yguidefont = font(24, "Helvetica Bold"),
    legendfont = font(20, "Helvetica Bold"),
    titlefont  = font(24, "Helvetica Bold"),
    tick_direction = :out,
    linewidth  = 2.0,
    framestyle = :box,
    grid       = false,
    dpi        = 600
)

println("Loading data...")

# ------------------------------------------------------------
# 1. Load CSV
# ------------------------------------------------------------
csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1.csv"

df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

# ------------------------------------------------------------
# 2. Load networks
# ------------------------------------------------------------
dir = raw"F:\HR_Project\noise1"

jld_path     = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]

println("Data loaded.")

# ------------------------------------------------------------
# 3. Feedback classifiers
# ------------------------------------------------------------
function classify_pos_feedback(mat)
    pfb = sum(mat[i,i] > 0 for i in 1:3) +
          sum((mat[i,j]*mat[j,i] > 0) for (i,j) in [(1,2),(1,3),(2,3)]) +
          (mat[1,2]*mat[2,3]*mat[3,1] > 0 ? 1 : 0) +
          (mat[1,3]*mat[3,2]*mat[2,1] > 0 ? 1 : 0)
    total_edges = count(x -> x != 0, mat)
    return pfb > 0 ? ("Pos FB Present", total_edges) : ("Pos FB Absent", total_edges)
end

function classify_neg_feedback(mat)
    nfb = sum(mat[i,i] < 0 for i in 1:3) +
          sum((mat[i,j]*mat[j,i] < 0) for (i,j) in [(1,2),(1,3),(2,3)]) +
          (mat[1,2]*mat[2,3]*mat[3,1] < 0 ? 1 : 0) +
          (mat[1,3]*mat[3,2]*mat[2,1] < 0 ? 1 : 0)
    total_edges = count(x -> x != 0, mat)
    return nfb > 0 ? ("Neg FB Present", total_edges) : ("Neg FB Absent", total_edges)
end

# ------------------------------------------------------------
# 4. Build DataFrames for Pos and Neg FB
# ------------------------------------------------------------
function build_feedback_df(all_matrices, net_counts, classifier)
    feedback_list = []
    for id in keys(all_matrices)
        mat = all_matrices[id].adj_matrix
        if all(mat .== 0)
            continue
        end
        cat, tot_edges = classifier(mat)
        idx             = findfirst(==(id), net_counts.net_id)
        multimodal_val  = isnothing(idx) ? 0.0 : net_counts[idx, :percentage_multimodal]
        multimodal_flag = multimodal_val > 0 ? 1 : 0
        push!(feedback_list, (net_id=id, category=cat, multimodal=multimodal_flag, total_edges=tot_edges))
    end
    return DataFrame(feedback_list)
end

feedback_df_pos = build_feedback_df(all_matrices, net_counts, classify_pos_feedback)
feedback_df_neg = build_feedback_df(all_matrices, net_counts, classify_neg_feedback)

println("DataFrames built.")

# ------------------------------------------------------------
# 5. Statistical analysis function with Custom Binning & Printing
# ------------------------------------------------------------
function run_fisher_analysis(feedback_df, present_label, absent_label, analysis_name)
    bins = [
        (min=1, max=1, label="1"),
        (min=2, max=3, label="2-3"),
        (min=4, max=5, label="4-5"),
        (min=6, max=7, label="6-7"),
        (min=8, max=9, label="8-9")
    ]

    bin_labels       = String[]
    p_one_list       = Float64[]
    p_two_list       = Float64[]
    log_or_list      = Float64[]
    total_net_counts = Int[]

    # Print table header
    println("\n" * "="^95)
    println("  STATISTICAL ANALYSIS TABLE: $analysis_name")
    println("="^95)
    @printf("%-6s | %-5s | %-5s | %-5s | %-5s | %-7s | %-8s | %-10s | %-10s | %-5s\n", 
            "Bin", "a", "b", "c", "d", "Total", "OR", "Log OR", "p (1-side)", "p (2-side)")
    println("-"^95)

    for b in bins
        subset = filter(row -> row.total_edges >= b.min && row.total_edges <= b.max, feedback_df)
        total_nets = nrow(subset)
        
        if total_nets == 0
            continue
        end

        a     = sum((subset.category .== present_label) .& (subset.multimodal .== 1))
        b_val = sum((subset.category .== present_label) .& (subset.multimodal .== 0))
        c     = sum((subset.category .== absent_label)  .& (subset.multimodal .== 1))
        d     = sum((subset.category .== absent_label)  .& (subset.multimodal .== 0))

        # Haldane-Anscombe correction for odds ratio if zeros are found
        a_corr = a == 0 ? a + 0.5 : Float64(a)
        b_corr = b_val == 0 ? b_val + 0.5 : Float64(b_val)
        c_corr = c == 0 ? c + 0.5 : Float64(c)
        d_corr = d == 0 ? d + 0.5 : Float64(d)

        OR_calc    = (a_corr * d_corr) / (b_corr * c_corr)
        logOR_calc = log(OR_calc)

        fet = FisherExactTest(Int(a), Int(b_val), Int(c), Int(d))
        p2  = pvalue(fet; tail=:both)
        
        # Calculate one-sided p-value
        p1_left  = pvalue(fet; tail=:left)
        p1_right = pvalue(fet; tail=:right)
        p1       = min(p1_left, p1_right)

        # Print the row values dynamically
        @printf("%-6s | %-5d | %-5d | %-5d | %-5d | %-7d | %-8.4f | %-10.4f | %-10.4e | %-10.4e\n", 
                b.label, a, b_val, c, d, total_nets, OR_calc, logOR_calc, p1, p2)

        push!(bin_labels,       b.label)
        push!(log_or_list,      logOR_calc)
        push!(p_one_list,       p1)
        push!(p_two_list,       p2)
        push!(total_net_counts, total_nets)
    end
    println("="^95)

    return bin_labels, log_or_list, p_one_list, p_two_list, total_net_counts
end

bin_labels_pos, log_or_pos, p_one_pos, p_two_pos, net_counts_pos =
    run_fisher_analysis(feedback_df_pos, "Pos FB Present", "Pos FB Absent", "Positive Feedback Analysis")

bin_labels_neg, log_or_neg, p_one_neg, p_two_neg, net_counts_neg =
    run_fisher_analysis(feedback_df_neg, "Neg FB Present", "Neg FB Absent", "Negative Feedback Analysis")

println("Fisher analysis complete.")

# ------------------------------------------------------------
# 6. Plot builder function
# ------------------------------------------------------------
function build_panel(bin_labels, log_or_list, p_one_list, p_two_list, total_net_counts, panel_title)

    n        = length(bin_labels)
    x_pos    = 1:n
    x_labels = (1:n, bin_labels)
    
    # --- SWAP TO ONE-SIDED P-VALUE HERE IF DESIRED ---
    # Changing this line to 'p_one_list' makes the scatter points reflect 1-sided significance
    #is_sig   = p_one_list .<= 0.05
    is_sig   = p_two_list .<= 0.05
    nonsig_idx = findall(.!is_sig)
    sig_idx    = findall(is_sig)

    # --- Main bar plot (Configured for Log Scale 10^0 to 10^3) ---
    p_main = bar(
        x_pos, total_net_counts,
        xlabel          = "Total Edges",
        ylabel          = "Total Network Count",
        yguidefontcolor = :darkblue,
        ytickfontcolor  = :darkblue,
        title           = panel_title,
        color           = :navy,
        alpha           = 0.7,
        xticks          = x_labels,
        xlim            = (0.5, n + 0.5),
        yscale          = :log10,
        ylim            = (10^0, 10^3),
        label           = "",
        legend          = false,
        framestyle      = :box,
        linewidth       = 2, 
        foreground_color_border = :black,
        left_margin     = 25Plots.mm,
        right_margin    = 25Plots.mm,
        top_margin      = 10Plots.mm,
        bottom_margin   = 14Plots.mm
    )

    # --- Twin axis (Configured for Log OR from -2 to 6) ---
    p_twin = twinx(p_main)

    plot!(p_twin,
        xlim            = (0.5, n + 0.5),
        ylim            = (-2.0, 6.0),
        ylabel          = "Log Odds Ratio",
        yguidefontcolor = :red,
        ytickfontcolor  = :red,
        xticks          = false,
        legend          = false,
        framestyle      = :box,
        linewidth       = 2,
        foreground_color_border = :black,
        ytickfont       = font(22, "Helvetica Bold"),
        yguidefont      = font(24, "Helvetica Bold")
    )

    # Scatter points tracking log OR values
    if !isempty(nonsig_idx)
        scatter!(p_twin, x_pos[nonsig_idx], log_or_list[nonsig_idx],
            color=:white, markerstrokecolor=:red, markerstrokewidth=2,
            markershape=:circle, markersize=10, label="")
    end
    if !isempty(sig_idx)
        scatter!(p_twin, x_pos[sig_idx], log_or_list[sig_idx],
            color=:red, markershape=:circle, markersize=10, label="")
    end

    hline!(p_twin, [0.0], linestyle=:dash, color=:black, linewidth=1.5, label="")

    # =======================================================
    # TICK LOGIC
    # =======================================================
    xl = Plots.xlims(p_main)
    yl = Plots.ylims(p_main)        
    yl_tw = Plots.ylims(p_twin)
    
    tl = 0.03 # Major tick length
    
    # REMOVED: Bottom X-ticks loops completely to clean up the plot area.

    # Left Y-ticks (Dark Blue) - Drawn on p_main
    yt_major = Plots.yticks(p_main)[1][1]
    dy_len = tl * (xl[2] - xl[1])
    for y in yt_major
        plot!(p_main, [xl[1], xl[1] + dy_len], [y, y], lc=:darkblue, lw=2, label="")
    end

    # Right Y-ticks (Red) - Drawn on p_twin 
    yt_tw_major = Plots.yticks(p_twin)[1][1]
    dy_len_tw = tl * (xl[2] - xl[1])
    for y in yt_tw_major
        plot!(p_twin, [xl[2], xl[2] - dy_len_tw], [y, y], lc=:red, lw=2, label="")
    end

    # ensure the box is perfectly square and thick
    plot!(p_main, foreground_color_border=:black, lw=2)
    plot!(p_twin, foreground_color_border=:black, lw=2)

    return p_main
end

# ------------------------------------------------------------
# 7. Build both panels
# ------------------------------------------------------------
p_pos = build_panel(
    bin_labels_pos, log_or_pos, p_one_pos, p_two_pos, net_counts_pos,
    "Pos FB Impact on HR-Enabling" 
)

p_neg = build_panel(
    bin_labels_neg, log_or_neg, p_one_neg, p_two_neg, net_counts_neg,
    "Neg FB Impact on HR-Enabling"
)

# ------------------------------------------------------------
# 8. Combine into 1×2 layout
# ------------------------------------------------------------
combined = plot(
    p_pos, p_neg,
    layout = (1, 2),
    size   = (2000, 700)
)

display(combined)

# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------
savefig(combined, "fb_combined_analysis.png")
savefig(combined, "fb_combined_analysis.svg")

println("Done. Saved fb_combined_analysis.png")

### Fig 2H : Contigency table & hist for iFFL, w/o iFFL

In [ ]:
###############################################################
# Full Analysis: Multimodality vs Incoherent Feed-Forward Loops (iFFL)
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using Plots
using HypothesisTests
using Colors
using Printf

gr()
Plots.default(dpi=600)

# ------------------------------------------------------------
# 1. Load multimodal CSV
# ------------------------------------------------------------
csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1.csv"
df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

# ------------------------------------------------------------
# 2. Load all networks from JLD2
# ------------------------------------------------------------
dir = raw"F:\HR_Project\noise1"
jld_path = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]
all_net_ids = collect(keys(all_matrices))

# ------------------------------------------------------------
# 3. Function to detect Incoherent Feed-Forward Loops (iFFL)
# ------------------------------------------------------------
function is_incoherent_ffl(mat)
    nodes = [1, 2, 3]
    for i in nodes
        for j in nodes
            if j == i; continue; end
            for k in nodes
                if k == i || k == j; continue; end
                if mat[j, i] != 0 && mat[k, j] != 0 && mat[k, i] != 0
                    if sign(mat[k, i]) != sign(mat[j, i] * mat[k, j])
                        return true
                    end
                end
            end
        end
    end
    return false
end

# ------------------------------------------------------------
# 4. Sanity checks on known iFFL examples
# ------------------------------------------------------------
test_iffl_1 = [0   0  0;
               1   0  0;
              -1   1  0]

test_iffl_2 = [0   0  0;
               1   0  0;
               1  -1  0]

test_cffl = [0  0  0;
             1  0  0;
             1  1  0]

@assert is_incoherent_ffl(test_iffl_1) == true  "Sanity check failed: iFFL type 1 not detected"
@assert is_incoherent_ffl(test_iffl_2) == true  "Sanity check failed: iFFL type 2 not detected"
@assert is_incoherent_ffl(test_cffl)   == false "Sanity check failed: cFFL incorrectly flagged as iFFL"
println("Sanity check passed: all three iFFL test cases correct")

# ------------------------------------------------------------
# 4a. Diagnostic: count iFFL networks across all non-zero networks
# ------------------------------------------------------------
n_iffl = sum([is_incoherent_ffl(all_matrices[id].adj_matrix) for id in all_net_ids if any(all_matrices[id].adj_matrix .!= 0)])
println("\nDiagnostic: iFFL count across all non-zero networks = ", n_iffl)

# ------------------------------------------------------------
# 5. Filter out all-zero networks and merge with CSV counts
# ------------------------------------------------------------
filtered_ids = [id for id in all_net_ids if any(all_matrices[id].adj_matrix .!= 0)]
all_nets_df = DataFrame(net_id = filtered_ids)

net_counts_full = leftjoin(all_nets_df, net_counts, on=:net_id)

net_counts_full.count = coalesce.(net_counts_full.count, 0)
net_counts_full.percentage_multimodal = coalesce.(net_counts_full.percentage_multimodal, 0.0)

# Use integer count for comparisons to avoid floating-point zero issues
net_counts_full.is_multimodal = net_counts_full.count .> 0

net_counts_full.has_iffl = [is_incoherent_ffl(all_matrices[id].adj_matrix) for id in net_counts_full.net_id]

# ------------------------------------------------------------
# 6. Contingency table counts
# ------------------------------------------------------------
n_multimodal_iffl        = sum(net_counts_full.is_multimodal .& net_counts_full.has_iffl)
n_multimodal_no_iffl     = sum(net_counts_full.is_multimodal .& .!net_counts_full.has_iffl)
n_non_multimodal_iffl    = sum(.!net_counts_full.is_multimodal .& net_counts_full.has_iffl)
n_non_multimodal_no_iffl = sum(.!net_counts_full.is_multimodal .& .!net_counts_full.has_iffl)

println("\nContingency Table Counts (With iFFL vs Without iFFL):")
println("Multimodal + With iFFL        = ", n_multimodal_iffl)
println("Multimodal + Without iFFL     = ", n_multimodal_no_iffl)
println("Non-Multimodal + With iFFL    = ", n_non_multimodal_iffl)
println("Non-Multimodal + Without iFFL = ", n_non_multimodal_no_iffl)

# ------------------------------------------------------------
# 7. Build 2x2 table
# ------------------------------------------------------------
a = n_multimodal_iffl
b = n_multimodal_no_iffl
c = n_non_multimodal_iffl
d = n_non_multimodal_no_iffl

table = [a b; c d]
println("\n2x2 Table:")
println(table)

# ------------------------------------------------------------
# 8. Odds Ratio with Haldane-Anscombe correction if any cell is zero
# ------------------------------------------------------------
a_or, b_or, c_or, d_or = Float64(a), Float64(b), Float64(c), Float64(d)
if any([a, b, c, d] .== 0)
    a_or += 0.5
    b_or += 0.5
    c_or += 0.5
    d_or += 0.5
end

OR    = (a_or * d_or) / (b_or * c_or)
logOR = log(OR)
se    = sqrt(1/a_or + 1/b_or + 1/c_or + 1/d_or)
CI_lo = exp(logOR - 1.96 * se)
CI_hi = exp(logOR + 1.96 * se)

println("\nOdds Ratio         = ", round(OR,    digits=4))
println("Log Odds Ratio     = ", round(logOR, digits=4))
println("95% CI             = (", round(CI_lo, digits=4), ", ", round(CI_hi, digits=4), ")")

# ------------------------------------------------------------
# 9. Fisher Exact Test
# ------------------------------------------------------------
fet   = FisherExactTest(a, b, c, d)
p_two = pvalue(fet; tail=:both)
p_one = pvalue(fet; tail=:right)

println("\nFisher Test Results:")
println("One-sided p-value (tail=right, OR > 1) = ", p_one)
println("Two-sided p-value                      = ", p_two)

# ------------------------------------------------------------
# 10. Manual Horizontal Stacked Bar Plot 
#     HR = Multimodal, Non-HR = Non-Multimodal
#     Uses the Fisher's test contingency counts a, b, c, d
# ------------------------------------------------------------
hr_total    = a + b   # multimodal total
nonhr_total = c + d   # non-multimodal total

hr_pct_fbl    = hr_total    == 0 ? 0.0 : (a / hr_total)    * 100   # multimodal w/ iFFL
nonhr_pct_fbl = nonhr_total == 0 ? 0.0 : (c / nonhr_total) * 100   # non-multimodal w/ iFFL

col_fbl    = colorant"#F08080"  # w/ iFFL  (red)
col_no_fbl = colorant"#808080"  # w/o iFFL (gray)

bar_h   = 0.6   # bar thickness
y_hr    = 2.0   # y-center for HR (Multimodal) bar (top)
y_nonhr = 1.0   # y-center for Non-HR (Non-Multimodal) bar (bottom)

function hbar_segment(y, x0, x1, h)
    Shape([x0, x1, x1, x0], [y-h/2, y-h/2, y+h/2, y+h/2])
end

plot2 = plot(
    legend = false,
    xlims  = (0, 100),
    ylims  = (0.3, 2.7),
    yticks = ([y_nonhr, y_hr], ["Non-HR", "HR"]),
    xticks = ([0, 100], ["0%", "100%"]),
    framestyle = :box,
    grid = false,
    size = (500, 400),
    xtickfont  = font(14, "Helvetica"),
    ytickfont  = font(14, "Helvetica"),
    bottom_margin = 5Plots.mm,
    left_margin   = 8Plots.mm,
    right_margin   = 8Plots.mm,
    xmirror = true   # puts 0%/100% labels on top
)

# HR (Multimodal) bar: red first (0 -> hr_pct_fbl), then gray (hr_pct_fbl -> 100)
plot!(plot2, hbar_segment(y_hr, 0, hr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_hr, hr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

# Non-HR (Non-Multimodal) bar: red first, then gray on top
plot!(plot2, hbar_segment(y_nonhr, 0, nonhr_pct_fbl, bar_h),
      color = col_fbl, linecolor = :black, linewidth = 0.5)
plot!(plot2, hbar_segment(y_nonhr, nonhr_pct_fbl, 100, bar_h),
      color = col_no_fbl, linecolor = :black, linewidth = 0.5)

display(plot2)

# ------------------------------------------------------------
# 11. SAVE Horizontal Stacked Bar Plot
# ------------------------------------------------------------
savefig(plot2, "figure_2H_style_iFFL.svg")
savefig(plot2, "figure_2H_style_iFFL.png")

println("Analysis complete. Stacked horizontal bar plot saved.")

In [ ]:
using Plots
using Colors

Plots.default(dpi=600)

# =========================================================
# DATA
# =========================================================
counts = [1625   1351;
           97  337]

p_val = "< 0.0001"
odds_ratio = 4.17

# =========================================================
# LOG TRANSFORM 
# =========================================================
log_counts = log10.(counts .+ 1)

# =========================================================
# AXES
# =========================================================
x = [1, 2]
y = [1, 2]

# =========================================================
# HEATMAP
# =========================================================
h = heatmap(
    x,
    y,
    log_counts,

    c = :YlGnBu, 
    colorbar = true,
    colorbar_pad = 15,  

    framestyle = :box,
    grid = false,
    linewidth = 2.0,
    aspect_ratio = 1,
    size = (750, 600),

    xlims = (0.5, 2.5),
    ylims = (0.5, 2.5),

    xticks = (x, ["With iFFL", "Without iFFL"]),
    yticks = (y, ["HR-Enabling \nNetworks", "HR-Disabling \nNetworks"]),
    yflip = true,

    xtickfont  = font(14, "Helvetica Bold"),
    ytickfont  = font(14, "Helvetica Bold"),
    xguidefont = font(18, "Helvetica Bold"),
    yguidefont = font(18, "Helvetica Bold"),
    legendfont = font(12, "Helvetica Bold"),

    left_margin   = 25Plots.mm,
    right_margin  = 20Plots.mm,   
    top_margin    = 15Plots.mm,
    bottom_margin = 15Plots.mm,

    colorbar_title = "\nlog10(count + 1)"
)

# =========================================================
# CELL ANNOTATIONS 
# =========================================================
# Text colors configured to dynamically contrast against YlGnBu tiles
annotate!(1, 1, text("n = 1625", 12, :white, :bold))
annotate!(2, 1, text("n = 1351", 12, :white, :bold))

annotate!(1, 2, text("n = 97", 12, :black, :bold))
annotate!(2, 2, text("n = 337", 12, :black, :bold))

# =========================================================
# DISPLAY & SAVE
# =========================================================
display(h)
savefig(h, "contingency_heatmap_iFFL.png")
savefig(h, "contingency_heatmap_iFFL.svg")

In [ ]:
###############################################################
# cFFL and iFFL Combined
###############################################################

using CSV
using DataFrames
using JLD2
using Statistics
using HypothesisTests
using Printf
using Plots
using Measures

gr()

Plots.default(
    xtickfont     = font(22, "Helvetica Bold"),
    ytickfont     = font(22, "Helvetica Bold"),
    xguidefont    = font(24, "Helvetica Bold"),
    yguidefont    = font(24, "Helvetica Bold"),
    legendfont    = font(20, "Helvetica Bold"),
    titlefont     = font(24, "Helvetica Bold"),
    tick_direction = :out,
    linewidth     = 2.0,
    framestyle    = :box,
    grid          = false,
    dpi           = 600,
    left_margin             = 25Plots.mm,
    right_margin            = 25Plots.mm,
    top_margin              = 10Plots.mm,
    bottom_margin           = 14Plots.mm
)

println("Loading data...")

# ------------------------------------------------------------
# 1. Load CSV
# ------------------------------------------------------------
csv_file = raw"F:\HR_Project\noise1\summary_clean_noise1.csv"
df = CSV.read(csv_file, DataFrame; header=true)
rename!(df, names(df)[1] => :net_id)

total_param_id = 1000
net_counts = combine(groupby(df, :net_id), nrow => :count)
net_counts.percentage_multimodal = (net_counts.count ./ total_param_id) .* 100

# ------------------------------------------------------------
# 2. Load networks
# ------------------------------------------------------------
dir = raw"F:\HR_Project\noise1"

jld_path     = joinpath(dir, "unique_networks_table.jld2")
network_data = load(jld_path)
all_matrices = network_data["network_table"]

println("Data loaded.")

# ------------------------------------------------------------
# 3. Motif Detectors
# ------------------------------------------------------------
function is_coherent_ffl(mat)
    nodes = [1, 2, 3]
    for i in nodes, j in nodes, k in nodes
        if j == i || k == i || k == j; continue; end
        if mat[j, i] != 0 && mat[k, j] != 0 && mat[k, i] != 0
            if sign(mat[k, i]) == sign(mat[j, i] * mat[k, j]); return true; end
        end
    end
    return false
end

function is_incoherent_ffl(mat)
    nodes = [1, 2, 3]
    for i in nodes, j in nodes, k in nodes
        if j == i || k == i || k == j; continue; end
        if mat[j, i] != 0 && mat[k, j] != 0 && mat[k, i] != 0
            if sign(mat[k, i]) != sign(mat[j, i] * mat[k, j]); return true; end
        end
    end
    return false
end

# ------------------------------------------------------------
# 4. Build Motif DataFrame
# ------------------------------------------------------------
function build_motif_df(all_matrices, net_counts)
    rows = []
    for id in keys(all_matrices)
        mat = all_matrices[id].adj_matrix
        if all(mat .== 0); continue; end
        idx = findfirst(==(id), net_counts.net_id)
        cnt = isnothing(idx) ? 0 : net_counts[idx, :count]
        push!(rows, (net_id=id, total_edges=count(x -> x != 0, mat), has_cffl=is_coherent_ffl(mat), has_iffl=is_incoherent_ffl(mat), is_multimodal=(cnt > 0 ? 1 : 0)))
    end
    return DataFrame(rows)
end

motif_df = build_motif_df(all_matrices, net_counts)

# ------------------------------------------------------------
# 5. Statistical analysis function
# ------------------------------------------------------------
function run_motif_fisher_analysis(motif_df, motif_col, analysis_name)
    bins = [(min=1, max=1, label="1"), (min=2, max=3, label="2-3"), (min=4, max=5, label="4-5"), (min=6, max=7, label="6-7"), (min=8, max=9, label="8-9")]
    bin_labels, log_or_list, p_two_list, total_net_counts = String[], Float64[], Float64[], Int[]

    println("\n" * "="^95 * "\n  STATISTICAL ANALYSIS TABLE: $analysis_name\n" * "="^95)
    @printf("%-6s | %-5s | %-5s | %-5s | %-5s | %-7s | %-8s | %-10s | %-10s\n", "Bin", "a", "b", "c", "d", "Total", "OR", "Log OR", "p (2-side)")
    println("-"^95)

    for b in bins
        subset = filter(row -> row.total_edges >= b.min && row.total_edges <= b.max, motif_df)
        total_nets = nrow(subset)
        if total_nets == 0; continue; end

        a = sum((subset[!, motif_col] .== true)  .& (subset.is_multimodal .== 1))
        b_val = sum((subset[!, motif_col] .== true)  .& (subset.is_multimodal .== 0))
        c = sum((subset[!, motif_col] .== false) .& (subset.is_multimodal .== 1))
        d = sum((subset[!, motif_col] .== false) .& (subset.is_multimodal .== 0))

        fet = FisherExactTest(Int(a), Int(b_val), Int(c), Int(d))
        p2 = pvalue(fet; tail=:both)
        
        a_val = (a == 0 ? 0.5 : Float64(a))
        b_val_c = (b_val == 0 ? 0.5 : Float64(b_val))
        c_val = (c == 0 ? 0.5 : Float64(c))
        d_val = (d == 0 ? 0.5 : Float64(d))
        OR = (a_val * d_val) / (b_val_c * c_val)
        
        @printf("%-6s | %-5d | %-5d | %-5d | %-5d | %-7d | %-8.4f | %-10.4f | %-10.4e\n", b.label, a, b_val, c, d, total_nets, OR, log(OR), p2)
        push!(bin_labels, b.label); push!(log_or_list, log(OR)); push!(p_two_list, p2); push!(total_net_counts, total_nets)
    end
    return bin_labels, log_or_list, p_two_list, total_net_counts
end

bl_c, lo_c, p2_c, cnt_c = run_motif_fisher_analysis(motif_df, :has_cffl, "cFFL Analysis")
bl_i, lo_i, p2_i, cnt_i = run_motif_fisher_analysis(motif_df, :has_iffl, "iFFL Analysis")

# ------------------------------------------------------------
# 6. Panel builder
# ------------------------------------------------------------
function build_panel(bin_labels, log_or_list, p_two_list, total_net_counts, panel_title, bar_color, scatter_color)
    n = length(bin_labels)
    is_sig = p_two_list .<= 0.05
    p_main = bar(1:n, total_net_counts, xlabel="Total Edges", ylabel="Total Network Count", yguidefontcolor=bar_color, title=panel_title, color=bar_color, alpha=0.7, xticks=(1:n, bin_labels), yscale=:log10, ylim=(10^0, 10^3), label="", legend=false, framestyle=:box)
    p_twin = twinx(p_main)
    plot!(p_twin, ylim=(-2.0, 6.0), ylabel="Log Odds Ratio", yguidefontcolor=scatter_color, label="")
    scatter!(p_twin, (1:n)[.!is_sig], log_or_list[.!is_sig], color=:white, markerstrokecolor=scatter_color, markerstrokewidth=2, markershape=:circle, markersize=10, label="")
    scatter!(p_twin, (1:n)[is_sig], log_or_list[is_sig], color=scatter_color, markershape=:circle, markersize=10, label="")
    hline!(p_twin, [0.0], linestyle=:dash, color=:black, label="")
    return p_main
end

p_cffl = build_panel(bl_c, lo_c, p2_c, cnt_c, "cFFL Impact on HR-Enabling", :darkred, colorant"#FF6347")
p_iffl = build_panel(bl_i, lo_i, p2_i, cnt_i, "iFFL Impact on HR-Enabling", :navy, colorant"#1F77B4")

combined = plot(p_cffl, p_iffl, layout=(1, 2), size=(2000, 700))
display(combined)

# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------
savefig(combined, "cffl_iffl_combined_analysis.png")
savefig(combined, "cffl_iffl_combined_analysis.svg")
println("Done. Saved figures.")